# 🧺 추세추종 포트폴리오 (Vol-Targeted Multi-Asset Trend)

`US_trend_following.ipynb`의 세 가지 확장을 **하나로 합친 실전형 버전**입니다.
개별 자산을 따로 보는 대신, **여러 자산의 추세 신호를 모아 하나의 포트폴리오**로 운용합니다.

## 무엇이 달라졌나 (3대 확장)
1. **다자산 분산(#2)** — 지수·주식·금·채권 9개 자산 중 **추세가 살아있는 것들만** 동시에 보유. 서로 다르게 움직이는 자산을 섞어 곡선을 부드럽게.
2. **변동성 타겟팅(#1)** — 각 자산 비중을 **변동성의 역수**로 배분(역변동성/리스크 패리티). 조용한 자산(채권)엔 많이, 사나운 자산(NVDA)엔 적게 → 한 자산이 리스크를 독식하지 않음. 총 노출은 레버리지 없이 ≤100%.
3. **레짐 필터(#3, 토글)** — SPY가 200일선 아래(위험장)면 전체를 꺼서 현금화하는 옵션.

## 왜 이게 핵심인가
개별 자산 추세추종은 "덜 잃지만 수익도 좀 낮은" 트레이드오프였다.
**분산 + 변동성 타겟팅**을 더하면, 자산 간 상쇄로 **낙폭이 더 줄고 Sharpe가 개선**되는 게 추세추종의 진짜 힘이다.
비교 벤치마크: 9자산 균등보유 · SPY 단독 보유 · 전통적 60/40(주식60·채권40).

## 엄격성 (동일 원칙)
- 신호·변동성 모두 **t시점까지 정보로만** 계산, 체결·비중은 다음 구간 반영(룩어헤드 방지)
- **편도 거래비용**을 리밸런스 회전율에 차감
- 파라미터는 **관습값 고정**(200일선·역변동성·월간 리밸런스) → 과적합 없음

> ⚠️ 투자 자문 아님. 교육/연구용.

In [ ]:
# =====================================================================
# ⚙️ Cell 1: 설정 & 임포트
# =====================================================================
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
try:
    plt.rcParams['font.family'] = 'Malgun Gothic'
except Exception:
    pass
plt.rcParams['axes.unicode_minus'] = False

TICKERS = ['SPY', 'QQQ', 'AAPL', 'MSFT', 'NVDA', 'JPM', 'XOM', 'GLD', 'TLT']
START, END = '2010-01-01', None
COST = 0.0005          # 편도 거래비용 0.05%
ANN  = 252

# 포트폴리오 파라미터 (모두 관습값 고정 -- 이 데이터에 맞추지 않음)
TREND_WIN  = 200       # 추세 필터: 200일 이동평균
VOL_WIN    = 63        # 변동성 추정 창(약 3개월)
RISK_FRAC  = 0.03      # 자산당 목표 변동성 기여(연 3%) -> 역변동성 비중
W_CAP      = 0.30      # 종목당 최대 비중
TARGET_LEV = 1.0       # 총 노출 상한(레버리지 금지 = 롱/현금)
REBAL      = 21        # 리밸런스 주기(영업일, 약 월간)

print(f'자산 {len(TICKERS)}개 | 추세 {TREND_WIN}일 | 리밸런스 {REBAL}일 | 종목상한 {W_CAP:.0%} | 총노출<={TARGET_LEV:.0%}')

In [ ]:
# =====================================================================
# 📥 Cell 2: 데이터 수집 (auto_adjust)
# =====================================================================
def fetch(t):
    df = yf.download(t, start=START, end=END, auto_adjust=True, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df['Close'].dropna()

closes = {}
for t in TICKERS:
    try:
        s = fetch(t)
        if len(s) > 300:
            closes[t] = s
            print(f'  OK  {t}: {len(s)}일')
        else:
            print(f'  SKIP {t}')
    except Exception as e:
        print(f'  FAIL {t}: {e}')
TICKERS = list(closes.keys())

# 공통 날짜로 정렬 (모든 자산이 존재하는 구간)
close = pd.DataFrame(closes).dropna()
print(f'\n공통 구간: {close.index[0].date()} ~ {close.index[-1].date()} ({len(close)}일), 자산 {list(close.columns)}')

In [ ]:
# =====================================================================
# 🧮 Cell 3: 신호/변동성/레짐 행렬 (모두 t시점까지 정보, 체결은 지연)
# =====================================================================
rets = close.pct_change().fillna(0.0)

# 추세 신호: 종가 > 200일 이동평균 이면 1(상승추세)
sma      = close.rolling(TREND_WIN).mean()
sig      = (close > sma).astype(float)
sig_exec = sig.shift(1).fillna(0.0)                 # 다음날 체결

# 변동성(연율화), 정보 지연
vol      = rets.rolling(VOL_WIN).std() * np.sqrt(ANN)
vol_exec = vol.shift(1)

# 레짐: SPY가 200일선 위인가 (위험장 필터, 옵션)
regime   = (close['SPY'] > sma['SPY']).shift(1).fillna(False)

print('신호/변동성/레짐 준비 완료. 평균 상승추세 자산 수:', round(sig_exec.sum(axis=1).mean(), 1), '/', len(TICKERS))

In [ ]:
# =====================================================================
# 🧺 Cell 4: 포트폴리오 백테스트 (역변동성 비중 + 총노출 상한 + 비용)
# =====================================================================
def build_portfolio(use_regime=False):
    w = pd.Series(0.0, index=TICKERS)
    port = pd.Series(0.0, index=close.index)
    exposure = pd.Series(0.0, index=close.index)
    turnover_total = 0.0
    dts = close.index
    for i in range(len(dts)):
        dt = dts[i]
        if i % REBAL == 0:
            neww = pd.Series(0.0, index=TICKERS)
            regime_ok = (not use_regime) or bool(regime.loc[dt])
            if regime_ok:
                for t in TICKERS:
                    v = vol_exec.loc[dt, t]
                    if sig_exec.loc[dt, t] == 1 and pd.notna(v) and v > 0:
                        neww[t] = min(RISK_FRAC / v, W_CAP)     # 역변동성 비중
                s = neww.sum()
                if s > TARGET_LEV:                              # 총노출 상한(레버리지 금지)
                    neww = neww * (TARGET_LEV / s)
            turn = float((neww - w).abs().sum())
            turnover_total += turn
            port.iloc[i] = float((w * rets.loc[dt]).sum()) - turn * COST
            w = neww
        else:
            port.iloc[i] = float((w * rets.loc[dt]).sum())
        exposure.iloc[i] = float(w.sum())
    return port, exposure, turnover_total

port_ret,        expo,        turn_tot        = build_portfolio(use_regime=False)
port_ret_regime, expo_regime, turn_tot_regime = build_portfolio(use_regime=True)

yrs = len(close) / ANN
print(f'추세 포트폴리오        | 연평균 회전율 {turn_tot/yrs:.2f} | 평균 노출 {expo.mean():.0%}')
print(f'추세 포트폴리오+레짐   | 연평균 회전율 {turn_tot_regime/yrs:.2f} | 평균 노출 {expo_regime.mean():.0%}')

In [ ]:
# =====================================================================
# 📊 Cell 5: 벤치마크 + 성과지표 비교
# =====================================================================
def perf(r):
    r = r.dropna(); eq = (1 + r).cumprod(); n = len(r)
    cagr = eq.iloc[-1] ** (ANN / n) - 1 if (n and eq.iloc[-1] > 0) else np.nan
    sharpe = r.mean() / r.std() * np.sqrt(ANN) if r.std() > 0 else 0.0
    dd = eq / eq.cummax() - 1; mdd = dd.min()
    calmar = cagr / abs(mdd) if mdd < 0 else np.nan
    return dict(CAGR=cagr, Vol=r.std()*np.sqrt(ANN), Sharpe=sharpe,
                MDD=mdd, Calmar=calmar, Final=float(eq.iloc[-1]))

# 벤치마크: 9자산 균등보유 / SPY 단독 / 60-40(SPY,TLT)
ew    = rets.mean(axis=1)
spy   = rets['SPY']
p6040 = 0.6 * rets['SPY'] + 0.4 * rets['TLT']

series = {
    '추세 포트폴리오'      : port_ret,
    '추세 포트폴리오+레짐' : port_ret_regime,
    '9자산 균등보유'       : ew,
    'SPY 단독보유'         : spy,
    '60/40 (SPY·TLT)'      : p6040,
}
tbl = pd.DataFrame({k: perf(v) for k, v in series.items()}).T
tbl = tbl[['CAGR', 'Vol', 'Sharpe', 'MDD', 'Calmar', 'Final']]
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
print('=== 성과 비교 (2010~현재, 비용 반영) ===')
print(tbl.to_string())
print('\n핵심 관전 포인트: 추세 포트폴리오의 MDD가 SPY/균등보유보다 얕고 Sharpe가 높은가?')

In [ ]:
# =====================================================================
# 📈 Cell 6: 자산곡선 · 낙폭 · 순노출(디레버리징) 시각화
# =====================================================================
fig, ax = plt.subplots(3, 1, figsize=(14, 12))

for k, v in series.items():
    eq = (1 + v.dropna()).cumprod()
    ax[0].plot(eq.index, eq.values, label=k,
               lw=2 if '추세' in k else 1.2,
               ls='-' if '추세' in k else '--')
ax[0].set_yscale('log'); ax[0].set_title('자산곡선 (로그 스케일)')
ax[0].legend(fontsize=9); ax[0].grid(alpha=0.3)

for k, v in series.items():
    eq = (1 + v.dropna()).cumprod(); dd = eq / eq.cummax() - 1
    ax[1].plot(dd.index, dd.values * 100, label=k,
               lw=2 if '추세' in k else 1.0)
ax[1].set_title('낙폭 (Drawdown, %) — 추세추종의 핵심 가치')
ax[1].legend(fontsize=9); ax[1].grid(alpha=0.3)

ax[2].fill_between(expo.index, 0, expo.values * 100, alpha=0.5, label='총 노출(%)')
ax[2].axhline(100, color='gray', ls=':', lw=1)
ax[2].set_title('추세 포트폴리오 순노출 — 위험장에서 현금으로 디레버리징하는지')
ax[2].set_ylabel('%'); ax[2].legend(fontsize=9); ax[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()

## 🧾 결과 해석 가이드

**핵심 질문: 분산 + 변동성 타겟팅이 개별 자산 추세추종보다, 그리고 단순 보유보다 나은가?**

| 봐야 할 것 | 성공 기준 |
|---|---|
| **MDD** | 추세 포트폴리오가 SPY·균등보유·60/40보다 **얕으면** 성공(위기 방어). |
| **Sharpe / Calmar** | 위험 대비 수익. 분산+역변동성으로 **가장 높게** 나오는 게 이상적. |
| **CAGR** | 강세장(2010~현재 미국 대세 상승)에선 SPY가 절대수익은 높을 수 있음 — 정상. 대신 변동성·낙폭이 큼. |
| **순노출 그래프(3번)** | 2020·2022 폭락 구간에서 노출이 **자동으로 내려가면** 변동성 타겟팅·추세필터가 제대로 작동하는 것. |

### 예상되는 정직한 결론
- 추세 포트폴리오는 **SPY보다 절대수익(CAGR)은 낮되, 변동성·MDD가 훨씬 작아 Sharpe/Calmar가 높음** — "적게 잃고 부드럽게".
- **레짐 필터**는 대개 MDD를 더 줄이지만 상승장 수익을 더 깎음(양날의 검).
- 60/40 대비로는 분산 자산군(금·채권 포함)이 더 많아 위기 방어가 강할 가능성.

### 이것이 이 프로젝트의 종착점
- **예측(ML) → 실패(비용 이기는 알파 없음)**
- **반응(규칙 추세추종) → 개별 자산 낙폭 방어**
- **분산 + 변동성 타겟팅(이 노트북) → 낙폭을 더 줄이고 위험 대비 수익 개선**

즉 "미래를 맞히려" 하지 않고, **리스크를 관리하며 추세에 규칙적으로 올라타는 것**이 가장 정직하고 견고한 접근이라는 결론.

### 더 나아간다면
- 파라미터(200/63/월간)를 **최적화하지 말 것** — 하는 순간 과적합. 견고성이 곧 가치.
- 실전화하려면: 세금·슬리피지 정교화, 리밸런스 밴드(드리프트 허용치), 자산군 확대(원자재·해외지수).